In [1]:
import math
import torch
import torchaudio
import torchaudio.transforms as T
import pandas as pd
from pathlib import Path
from tqdm import tqdm

### Setup

In [2]:
# Paths to data and output directories
DEV_AUDIO  = Path("data/FSD50K.dev_audio")
EVAL_AUDIO = Path("data/FSD50K.eval_audio")
GT_DEV     = Path("data/FSD50K.ground_truth/dev.csv")
GT_EVAL    = Path("data/FSD50K.ground_truth/eval.csv")
OUT_DIR    = Path("preprocessed")

In [3]:
#configuration: setting it such that I am only saving a few spectrograms for each set while developing the code, to save time.
SUBSET = False        # set to False to process everything, when code is ready
N_TRAIN = 500        # clips from train
N_VAL   = 10         # clips from val
N_EVAL  = 10         # clips from eval

#pull csv data
dev_csv  = pd.read_csv(GT_DEV)
eval_csv = pd.read_csv(GT_EVAL)

train_rows = dev_csv[dev_csv["split"] == "train"]
val_rows   = dev_csv[dev_csv["split"] == "val"]
eval_rows  = eval_csv

if SUBSET:
    train_rows = train_rows.head(N_TRAIN)
    val_rows   = val_rows.head(N_VAL)
    eval_rows  = eval_rows.head(N_EVAL)

print(f"Processing: {len(train_rows)} train | {len(val_rows)} val | {len(eval_rows)} eval")

Processing: 36796 train | 4170 val | 10231 eval


In [4]:
# Audio settings - These are just decided based on human hearing range and common settings in the literature, but can be changed if needed.
SAMPLE_RATE = 22050
N_MELS      = 64
HOP_LENGTH  = 256
N_FFT       = 1024
F_MIN       = 100
F_MAX       = 10000

### necessary functions

In [5]:
#define transform to mel spectrogram and to decibels
mel_transform = T.MelSpectrogram(
    sample_rate = SAMPLE_RATE,
    n_fft       = N_FFT,
    hop_length  = HOP_LENGTH,
    n_mels      = N_MELS,
    f_min       = F_MIN,
    f_max       = F_MAX,
    power       = 2.0,
)
to_db = T.AmplitudeToDB(stype="power", top_db=80.0)

#define function that processes audio and returns a log-mel spectrogram
def process_audio(path: Path) -> torch.Tensor:
    """Load a .wav and return an unnormalized log-mel spectrogram."""
    waveform, sr = torchaudio.load(path)
    waveform = torchaudio.functional.resample(waveform, sr, SAMPLE_RATE)
    waveform = waveform.mean(dim=0, keepdim=True)   # stereo → mono
    return to_db(mel_transform(waveform))            # (1, 128, time_frames)

#function for saving spectrograms to disk
def save_split(rows, audio_dir, split_name):
    out = OUT_DIR / split_name
    out.mkdir(parents=True, exist_ok=True)
    for _, row in tqdm(rows.iterrows(), total=len(rows), desc=f"Saving {split_name}"):
        spec = process_audio(audio_dir / f"{row['fname']}.wav")
        torch.save(spec, out / f"{row['fname']}.pt")


### first pass through data: compute mean and std

In [6]:
#compute mean and std for later normalization (this is just for the training set, as the val and eval sets should be normalized using the same parameters)
#this just runs trough the training set, computes the spectrograms, and keeps track of the sum, squared sum, and count of all values to compute the mean and std at the end. This is done in a memory-efficient way without storing all spectrograms in memory at once
print("\nPass 1: computing mean and std from training set...")
total_sum, total_sq_sum, total_count = 0.0, 0.0, 0

for _, row in tqdm(train_rows.iterrows(), total=len(train_rows)):
    spec = process_audio(DEV_AUDIO / f"{row['fname']}.wav")
    total_sum    += spec.sum().item()
    total_sq_sum += (spec ** 2).sum().item()
    total_count  += spec.numel()

NORM_MEAN = total_sum / total_count
NORM_STD  = (total_sq_sum / total_count - NORM_MEAN ** 2) ** 0.5
print(f"\nNORM_MEAN = {NORM_MEAN:.4f}")
print(f"NORM_STD  = {NORM_STD:.4f}")
print("→ Copy these two numbers into your Dataset class!")


Pass 1: computing mean and std from training set...


100%|██████████| 36796/36796 [19:28<00:00, 31.49it/s]  


NORM_MEAN = -17.9358
NORM_STD  = 20.7367
→ Copy these two numbers into your Dataset class!


### second pass through data: save pt files to disk

In [7]:
#run trough all sets and save spectrograms to disk as .pt files (this is done in a memory-efficient way without storing all spectrograms in memory at once)
print("\nPass 2: saving .pt files...")
save_split(train_rows, DEV_AUDIO,  "train")
save_split(val_rows,   DEV_AUDIO,  "val")
save_split(eval_rows,  EVAL_AUDIO, "eval")

print("\nDone! Folder structure:")
for split in ["train", "val", "eval"]:
    files = list((OUT_DIR / split).glob("*.pt"))
    print(f"  preprocessed/{split}/  →  {len(files)} files")


Pass 2: saving .pt files...


Saving train:   0%|          | 0/36796 [00:00<?, ?it/s]

Saving eval: 100%|██████████| 10231/10231 [07:13<00:00, 23.60it/s]



Done! Folder structure:
  preprocessed/train/  →  36796 files
  preprocessed/val/  →  4170 files
  preprocessed/eval/  →  10231 files
